In [2]:
import pandas as pd
import random
from pycaret.classification import *
import numpy as np

In [1]:
def add_class_col(df):
    labels = []
    for region in df["non_rep_region"]:
        if region[-3:] == "pos":
            labels.append(1)
        elif region[-3:] == "neg":
            labels.append(0)
    df["Class"] = labels

In [6]:
def setup_data():
    validate_data = pd.read_csv("experiment_2\\input_data\\validate.csv")
    train_data = pd.read_csv("experiment_2\\input_data\\train.csv")
    validate_data = validate_data.drop("non_rep_region", axis=1)
    train_data = train_data.drop("non_rep_region", axis=1)

    data = pd.concat([validate_data, train_data], ignore_index=True)
    print(data['acr_label'].value_counts())
    print("Data read")

    s = setup(data=data, target="acr_label", test_data=validate_data, index=False, verbose=False, fold=2)
    print("setup done")
    return s

In [ ]:
#compare models
setup_data()
best = compare_models()

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,09:36:09
Status,. . . . . . . . . . . . . . . . . .,Compiling Final Models
Estimator,. . . . . . . . . . . . . . . . . .,Gradient Boosting Classifier


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.6430,0.7004,0.6544,0.6399,0.6470,0.2860,0.2861,1973.4300
lightgbm,Light Gradient Boosting Machine,0.6401,0.6965,0.6427,0.6394,0.6410,0.2801,0.2801,120.1950
rf,Random Forest Classifier,0.6312,0.6772,0.6491,0.6268,0.6377,0.2625,0.2627,27.9850
et,Extra Trees Classifier,0.6290,0.6758,0.6560,0.6224,0.6387,0.2580,0.2584,25.2300
ada,Ada Boost Classifier,0.6177,0.6671,0.6011,0.6217,0.6112,0.2354,0.2355,423.0900
lr,Logistic Regression,0.5774,0.6016,0.5735,0.5780,0.5757,0.1548,0.1548,142.6150
knn,K Neighbors Classifier,0.5717,0.5939,0.5593,0.5735,0.5663,0.1433,0.1434,156.9400
nb,Naive Bayes,0.5703,0.5812,0.4967,0.5825,0.5362,0.1407,0.1422,18.2050
svm,SVM - Linear Kernel,0.5582,0.6367,0.7989,0.5501,0.6373,0.1164,0.1554,28.6900
dt,Decision Tree Classifier,0.5533,0.5507,0.5557,0.5530,0.5543,0.1065,0.1065,142.0850


Processing:   0%|          | 0/61 [00:00<?, ?it/s]

In [ ]:
#tune model
setup_data("rand")
model = create_model("gbc", verbose=False)
tuned = tune_model(model, optimize="F1")
tuned

In [ ]:
#get feature importance
s = setup_data("up")
features = s._fxs['Numeric']
models = ["lightgbm", "gbc", "ada", "rf", "et"]
frequency = {}
for i in range(5):
    for model in models:
        model_ob = create_model(model, verbose=False)
        try:
            importance = model_ob.feature_importances_
        except:
            print(f"{model} doesn't have feature importance")
            continue
        importance_args = np.argsort(importance)[:5]
        for indx in importance_args:
            frequency[indx] = frequency.get(indx, 0) + 1
    print(f"Iteration {i} complete")
for indx, freq in frequency.items():
    print(f"{features[indx]} {freq / 5}")
print(frequency)

